---

# 🐳 Partie 2 : Distribution avec Docker + PySpark

## Objectif

Transformer le système de recommandation en une application distribuée et containerisée :
- **Isolation** : Chaque étape dans un conteneur Docker indépendant
- **Scalabilité** : Traitement distribué avec PySpark (RDD + MapReduce)
- **Orchestration** : docker-compose pour gérer les 3 conteneurs
- **Partage de données** : Volume partagé entre les conteneurs

---

## Architecture de la solution

### 🏗️ Structure des conteneurs

```
partie2/
├── docker-compose.yml          ← Orchestration des 3 conteneurs
├── .env                        ← Configuration centralisée
│
├── acquisition/                ← Conteneur 1 : Téléchargement d'images
│   ├── Dockerfile
│   ├── requirements.txt
│   └── acquisition.py          ← Script PySpark
│
├── analysis/                   ← Conteneur 2 : Labeling + Profils
│   ├── Dockerfile
│   ├── requirements.txt
│   └── analysis.py             ← Script PySpark
│
└── recommendation/             ← Conteneur 3 : ML + Recommandations
    ├── Dockerfile
    ├── requirements.txt
    └── recommendation.py       ← Script PySpark
```

### 🔄 Flux de traitement

1. **Container acquisition** démarre → télécharge les images → partage via volume
2. **Container analysis** démarre → labeling + profils → partage JSON
3. **Container recommendation** démarre → ML + tests → génère les recommandations

Chaque conteneur utilise PySpark pour distribuer les calculs sur des RDDs (Resilient Distributed Datasets).

---

## 🔧 Implémentation PySpark - Exemples

### Exemple 1 : Acquisition d'images avec Map-Reduce

**Transformation Map** : Téléchargement distribué des images

In [ ]:
# Extrait de partie2/acquisition/acquisition.py
# ============================================

from pyspark import SparkContext, SparkConf

# Fonction de traitement (exécutée en parallèle sur chaque nœud)
def process_image(item: tuple) -> tuple:
    """
    Télécharge et sauvegarde une image (Map).
    Prend un tuple (counter, img_data, query) — toutes les données
    nécessaires passées dans le tuple pour éviter les problèmes de closure Spark.
    """
    counter, img_data, query = item
    filename = f"image_{counter:03d}.jpg"
    image_url = img_data['urls']['regular']

    try:
        response = requests.get(image_url, timeout=15)
        response.raise_for_status()

        filepath = IMAGES_DIR / filename
        filepath.write_bytes(response.content)

        with Image.open(filepath) as img:
            width, height = img.size

        return (filename, {"filename": filename, "width": width,
                           "height": height, "query": query, ...})
    except:
        return (None, None)

# Map-Reduce : Distribution du téléchargement
# all_images_data = list of tuples (counter, img_data, query)
images_rdd = sc.parallelize(all_images_data)        # Création du RDD
metadata_rdd = images_rdd.map(process_image)        # Map : traitement distribué
successful_rdd = metadata_rdd.filter(lambda x: x[0] is not None)  # Filter
results = successful_rdd.collect()                  # Collect : récupération

all_metadata = {filename: metadata for filename, metadata in results}
print(f"✅ {len(all_metadata)} images téléchargées")


### Exemple 2 : Labeling distribué des images

**Transformation Map** : Extraction de caractéristiques pour chaque image

In [ ]:
# Extrait de partie2/analysis/analysis.py
# ==========================================

def process_image_labels(metadata):
    """
    Extrait les caractéristiques d'une image (Map).
    Exécuté en parallèle sur chaque image.
    """
    filename = metadata['filename']
    filepath = f"{images_folder}/{filename}"
    
    # Extraction des couleurs dominantes avec KMeans
    img = Image.open(filepath)
    pixels = np.array(img).reshape(-1, 3)
    kmeans = KMeans(n_clusters=3, random_state=42)
    kmeans.fit(pixels)
    dominant_colors = kmeans.cluster_centers_.astype(int)
    
    # Détection de l'orientation
    width, height = img.size
    orientation = "paysage" if width > height else "portrait"
    
    return {
        'filename': filename,
        'colors': [rgb_to_color_name(tuple(color)) for color in dominant_colors],
        'orientation': orientation,
        'tags': metadata['tags'],
        'dimensions': {'width': width, 'height': height}
    }

# Map-Reduce : Labeling distribué
metadata_rdd = sc.parallelize(images_metadata)
labels_rdd = metadata_rdd.map(process_image_labels)  # Map distribué
all_labels = labels_rdd.collect()  # Reduce : collecte

# Sauvegarde
with open(f"{data_folder}/images_labels.json", 'w') as f:
    json.dump(all_labels, f, indent=2)

print(f"✅ {len(all_labels)} images labellisées")

### Exemple 3 : Calcul de recommandations distribué

**Map** : Score de pertinence pour chaque image / **Reduce** : Top 5

In [ ]:
# Extrait de partie2/recommendation/recommendation.py
# ===================================================

def compute_recommendation_score(item: tuple) -> tuple:
    """
    Calcule le score de pertinence pour une image (Map).
    Toutes les données nécessaires sont passées dans le tuple pour éviter
    les problèmes de closure avec Spark (le modèle ne peut pas être global).
    """
    filename, proba, idx, favorites, labels, user_profile = item

    # Exclure les favoris
    if filename in favorites:
        return None

    # Générer une raison basée sur le profil
    img_labels = labels[filename]
    reasons = []

    matching_colors = set(img_labels['color_names']) & set(user_profile['favorite_colors'])
    if matching_colors:
        reasons.append(f"couleurs {', '.join(matching_colors)}")

    if img_labels['orientation'] == user_profile['favorite_orientation']:
        reasons.append(f"orientation {img_labels['orientation']}")

    matching_tags = set(img_labels['tags']) & set(user_profile['favorite_tags'])
    if matching_tags:
        reasons.append(f"tags {', '.join(list(matching_tags)[:2])}")

    reason = f"Correspond à vos préférences : {' + '.join(reasons or ['profil général'])}"
    return (filename, float(proba), reason)


# Pour chaque utilisateur
for user_id in users.keys():
    # Entraînement d'un modèle Random Forest personnalisé
    model, y, idx_test, accuracy = train_user_model(user_id, users, df_features, X_all)

    # Prédire les probabilités pour toutes les images
    probas = model.predict_proba(X_all)[:, 1]
    favorites = set(users[user_id]['favorite_images'])
    user_profile = users[user_id]

    # Construire les items avec toutes les données nécessaires dans chaque tuple
    items = [(filename, proba, idx, favorites, labels, user_profile)
             for idx, (filename, proba) in enumerate(zip(df_features['filename'], probas))]

    # Map-Reduce : calcul des scores distribué
    items_rdd = sc.parallelize(items)
    scores_rdd = items_rdd.map(compute_recommendation_score)         # Map

    # Filter : retirer les None (favoris)
    valid_scores_rdd = scores_rdd.filter(lambda x: x is not None)

    # SortBy + Take : top 5
    top_recommendations = valid_scores_rdd.sortBy(
        lambda x: x[1],     # Trier par score
        ascending=False      # Décroissant
    ).take(5)                # Top 5

    all_recommendations[user_id] = [
        {"filename": fn, "score": score, "reason": reason}
        for fn, score, reason in top_recommendations
    ]

# Sauvegarde
with open(RECOMMENDATIONS_FILE, "w", encoding="utf-8") as f:
    json.dump(all_recommendations, f, ensure_ascii=False, indent=2)


---

## 🐳 Configuration Docker

### Dockerfile (exemple : acquisition)

Chaque conteneur utilise un Dockerfile similaire avec :
- Image de base : `python:3.10-slim`
- Installation de Java (requis pour PySpark) : `openjdk-21-jdk-headless`
- Installation des dépendances Python depuis `requirements.txt`
- Configuration de `JAVA_HOME` pour PySpark

In [ ]:
# Extrait de partie2/acquisition/Dockerfile

FROM python:3.10-slim

# Installer Java (requis pour PySpark)
RUN apt-get update && \
    apt-get install -y openjdk-21-jdk-headless && \
    rm -rf /var/lib/apt/lists/*

# Définir JAVA_HOME et PATH
ENV JAVA_HOME=/usr/lib/jvm/java-21-openjdk-amd64
ENV PATH=$PATH:$JAVA_HOME/bin

WORKDIR /app

# Copier et installer les dépendances
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copier le script
COPY acquisition.py .

# Lancer le script
CMD ["python", "acquisition.py"]


### docker-compose.yml

Orchestration des 3 conteneurs avec dépendances séquentielles :

In [ ]:
# partie2/docker-compose.yml

services:
  acquisition:
    container_name: projet_acquisition
    build: ./acquisition
    environment:
      - OUTPUT_DIR=/shared_data
      - UNSPLASH_ACCESS_KEY=${UNSPLASH_ACCESS_KEY}  # lu depuis .env
    volumes:
      - shared_data:/shared_data  # Volume partagé
    networks:
      - projet_network

  analysis:
    container_name: projet_analysis
    build: ./analysis
    environment:
      - INPUT_DIR=/shared_data
      - OUTPUT_DIR=/shared_data
    volumes:
      - shared_data:/shared_data
    networks:
      - projet_network
    depends_on:
      acquisition:
        condition: service_completed_successfully  # Attend acquisition

  recommendation:
    container_name: projet_recommendation
    build: ./recommendation
    environment:
      - INPUT_DIR=/shared_data
      - OUTPUT_DIR=/shared_data
    volumes:
      - shared_data:/shared_data
    networks:
      - projet_network
    depends_on:
      analysis:
        condition: service_completed_successfully  # Attend analysis

volumes:
  shared_data:  # Volume partagé entre tous les conteneurs

networks:
  projet_network:
    driver: bridge


---

## 🚀 Exécution de la Partie 2

### Commandes d'exécution (PowerShell)

```powershell
# 1. Se placer dans le dossier partie2
cd "c:\Users\maxen\.vscodeProject\DonneMassiv\fr\Projet\partie2"

# 2. (Optionnel) Supprimer un éventuel volume résiduel pour repartir de zéro
#    ⚠️ Cela supprime toutes les données du volume shared_data
docker compose down -v

# 3. Lancer les 3 conteneurs (build + exécution séquentielle)
docker compose up --build

# 4. Récupérer les données générées dans ./output
docker cp projet_recommendation:/shared_data ./output

# 5. Vérifier les fichiers générés
Get-ChildItem -Path ./output -Recurse | Select-Object Name, Length
```

> **Pourquoi `docker compose down -v` avant de relancer ?**  
> Le volume Docker `shared_data` persiste entre les exécutions. Sans le supprimer, les données d'un run précédent restent dans le volume. Pour garantir que la Partie 2 génère bien tout de zéro (sans hériter de la Partie 1), il faut purger le volume avant de relancer.

### Durée d'exécution

- **Build des images Docker** : ~3-4 minutes (installation Java + PySpark)
- **Acquisition** : ~5-10 minutes (téléchargement de ~120 images)
- **Analysis** : ~5-10 minutes (labeling + visualisations)
- **Recommendation** : ~2-5 minutes (ML + tests)
- **Total** : ~15-25 minutes

---


## 📊 Résultats de l'exécution

### Logs d'exécution (extraits)

**Container acquisition** :
```
✅ 120 images téléchargées avec succès
💾 Métadonnées sauvegardées : /shared_data/images_metadata.json
```

**Container analysis** :
```
✅ 120 images labellisées
✅ 5 utilisateurs analysés avec profils complets
💾 Visualisations sauvegardées : /shared_data/visualisations.png
```

**Container recommendation** :
```
🎯 Modèle pour user_001 : Précision : 88.89%
🎯 Modèle pour user_002 : Précision : 80.56%
🎯 Modèle pour user_003 : Précision : 83.33%
🎯 Modèle pour user_004 : Précision : 88.89%
🎯 Modèle pour user_005 : Précision : 88.89%

✅ Recommandations générées pour 5 utilisateurs
💾 Recommandations sauvegardées : /shared_data/recommendations.json

📝 Test 1 : Intégrité des données - ✅ RÉUSSI
📝 Test 2 : Qualité des recommandations - ✅ RÉUSSI
   Pertinence : 100.0% des recommandations correspondent au profil

🎉 TOUS LES TESTS RÉUSSIS !
```

### Fichiers générés (Partie 2)

Les fichiers sont copiés depuis le volume Docker vers `partie2/output/` via :
```powershell
docker cp projet_recommendation:/shared_data ./output
```

```
partie2/output/
├── images/                     ← ~120 images (~27 MB)
├── images_metadata.json        ← ~90 Ko (métadonnées complètes)
├── images_labels.json          ← ~68 Ko (labels + couleurs)
├── users.json                  ← ~3.5 Ko (5 profils utilisateur)
├── recommendations.json        ← ~4 Ko (top 5 par utilisateur)
└── visualisations.png          ← Graphiques de synthèse
```

---


## 📝 Conclusion - Partie 2

### Objectifs atteints

✅ **Conteneurisation** : 3 conteneurs Docker indépendants et isolés  
✅ **Distribution** : Utilisation de PySpark avec RDD et transformations Map-Reduce  
✅ **Orchestration** : docker-compose gérant les dépendances séquentielles  
✅ **Partage de données** : Volume Docker partagé entre les conteneurs  
✅ **Performances** : Traitement distribué sur 120 images (~20 minutes total)  
✅ **Tests** : Tous les tests de validation réussis (intégrité + qualité)  

### Comparaison Partie 1 vs Partie 2

| Aspect | Partie 1 (Notebook) | Partie 2 (Docker + PySpark) |
|--------|---------------------|------------------------------|
| **Environnement** | Local Python | Conteneurs Docker isolés |
| **Traitement** | Séquentiel | Distribué (RDD + MapReduce) |
| **Scalabilité** | Limitée | Haute (parallélisation) |
| **Déploiement** | Manuel | Automatisé (docker-compose) |
| **Isolation** | Faible | Forte (conteneurs) |
| **Reproductibilité** | Moyenne | Excellente |

### Technologies utilisées

- **Docker** : Conteneurisation + isolation
- **Docker Compose** : Orchestration multi-conteneurs
- **PySpark 3.5.3** : Framework de traitement distribué
- **Java 21** : Runtime pour PySpark
- **Python 3.10** : Langage de programmation
- **Volumes Docker** : Partage de données persistant

### Points clés de l'implémentation

🔧 **Map-Reduce** appliqué à toutes les étapes :
- Téléchargement d'images en parallèle (.map + .filter)
- Labeling distribué avec KMeans
- Calcul de scores de recommandation distribué (.sortBy + .take)

🐳 **Bonnes pratiques Docker** :
- Image de base légère (python:3.10-slim)
- Multi-stage non nécessaire (scripts simples)
- Variables d'environnement via .env
- Nettoyage des caches apt/pip

---

## 🎓 Synthèse finale du projet

Ce projet a démontré la transformation complète d'un système de recommandation :

1. **Partie 1** : Prototype fonctionnel en notebook Jupyter avec ML
2. **Partie 2** : Application distribuée containerisée avec PySpark

L'architecture finale est **scalable**, **reproductible**, et **déployable** dans un environnement de production.

---

**⚠️ IMPORTANT** : Avant de soumettre, assurez-vous de :
1. Ne PAS inclure le dossier `images/` dans le ZIP (trop volumineux)
2. Vérifier que tous les tests passent (Partie 1 + Partie 2)
3. Inclure tous les fichiers Docker (Dockerfiles, docker-compose.yml)
4. Créer le rapport de synthèse (4 pages PDF)

---